<a href="https://colab.research.google.com/github/mrbedoya/ml-2026/blob/main/11_ArbolesDecisionRandomForestClasificacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Árboles de Decisión Random Forest (Clasificación)

## 1. Librerias y configuraciones previas


In [79]:
# Tratamiento de datos
# ==============================================================================
import pandas as pd
import numpy as np

# Almacenar en caché los resultados de funciones en el disco
# ==============================================================================
import joblib

# Gestion de librerias
# ==============================================================================
from importlib import reload

# Matemáticas y estadísticas
# ==============================================================================
import math

# Preprocesado y modelado
# ==============================================================================

#Separar los datos entrenamiento y prueba
from sklearn.model_selection import train_test_split

#Escalar Variables
from sklearn.preprocessing import MinMaxScaler

#Configuracion de hiperparámetros
from sklearn.model_selection import ParameterGrid, GridSearchCV, RandomizedSearchCV

#Creación de modelo
from sklearn.ensemble import RandomForestClassifier

# Evaluación del modelo
from sklearn.metrics import classification_report

# Gráficos
# ==============================================================================
import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns

# Interuactuar con os y solicitudes desde HTTP
# ==============================================================================
import os
import sys
import requests

# Configuración warnings
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')


In [80]:
# Parametros globales
# ==============================================================================

# URL base de GitHub para el repositorio
GITHUB_BASE_URL = 'https://raw.githubusercontent.com/mrbedoya/ml-2026/main/'

# Nombres de los archivos requeridos
DATASET_FILE_NAME = '02_GermanCredit_Prep.csv'
UTILS_FILE_NAME = 'funciones.py'

# Nombres de los directorios para utilidades, datasets y modelos
DATASETS_PATH = 'datasets/'
UTILS_PATH = 'utils/'
MODELS_PATH = 'modelos/'
MODELS_CLASIFICACION_PATH = os.path.join(MODELS_PATH, 'clasificacion')
MODELS_SCALER_PATH = os.path.join(MODELS_PATH, 'scaler')

# URL completa del dataset y utils en GitHub
github_dataset_url = os.path.join(GITHUB_BASE_URL, DATASETS_PATH, DATASET_FILE_NAME)
github_utils_url = os.path.join(GITHUB_BASE_URL, UTILS_PATH, UTILS_FILE_NAME)

# Rutas locales completas de datasets y utils
local_dataset_path = os.path.join(DATASETS_PATH, DATASET_FILE_NAME)
local_utils_path = os.path.join(UTILS_PATH, UTILS_FILE_NAME)

In [81]:
## Cargar datos desde Google Drive
## ==============================================================================

#from google.colab import drive

## Montar Google Drive
#drive.mount('/gdrive')

## Establece el directorio de trabajo dentro de directorio espefíficado en Google Drive
#os.chdir("/gdrive/MyDrive/01_Academia/02_Cursos/20260401_ApredinzajeAutomáticoI_UdeA/")

## Listar archivos de directorio
##!ls

In [82]:
# Funcion local para la carga de archivos
# ==============================================================================

def download_file(url, path, description):
    '''
    Descarga un archivo desde una URL y lo guarda localmente.

    Args:
        url (str): URL del archivo
        path (str): Ruta local de destino
        description (str): Descripción del archivo
    Returns:
        None
    '''
    print(f"Descargando {description} desde: {url}")

    # Realiza una solicitud HTTP GET a la URL proporcionada
    response = requests.get(url)

    # Lanza una excepción si la descarga falla (código de estado HTTP no 200)
    response.raise_for_status()

    # Abre el archivo local en modo de escritura binaria ('wb')
    with open(path, 'wb') as f:
        # Escribe el contenido binario de la respuesta HTTP en el archivo local
        f.write(response.content)

    # Imprime un mensaje confirmando que la descarga fue exitosa y la ruta de guardado
    print(f"{description} descargado exitosamente a: {path}")

In [83]:
## Configuración entorno Colab
# ==============================================================================

# Creación de directorios locales
os.makedirs(DATASETS_PATH, exist_ok=True)
os.makedirs(UTILS_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

# Crear subdirectorios específicos para modelos y escalados
os.makedirs(MODELS_CLASIFICACION_PATH, exist_ok=True)
os.makedirs(MODELS_SCALER_PATH, exist_ok=True)

# Descarga de archivos
download_file(github_dataset_url, local_dataset_path, "dataset")
download_file(github_utils_url, local_utils_path, "funciones")

Descargando dataset desde: https://raw.githubusercontent.com/mrbedoya/ml-2026/main/datasets/02_GermanCredit_Prep.csv
dataset descargado exitosamente a: datasets/02_GermanCredit_Prep.csv
Descargando funciones desde: https://raw.githubusercontent.com/mrbedoya/ml-2026/main/utils/funciones.py
funciones descargado exitosamente a: utils/funciones.py


## 2. Funciones

In [84]:
## Funciones externas
# ==============================================================================

## Añade la ruta definida en 'UTILS_PATH' al 'sys.path' para que Python pueda encontrar módulos
#sys.path.append(UTILS_PATH)

## Importa Funciones externas
#from funciones import multiple_plot, plot_roc_curve

## 3. Carga del dataset

In [85]:
#Se crea un dataframe d con los datos obtenidos de archivo de entrada
d=pd.read_csv(local_dataset_path)

In [86]:
d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1120 entries, 0 to 1119
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   checking_account_status   1120 non-null   object
 1   loan_duration_mo          1120 non-null   int64 
 2   credit_history            1120 non-null   object
 3   purpose                   1120 non-null   object
 4   loan_amount               1120 non-null   int64 
 5   savings_account_balance   1120 non-null   object
 6   time_employed_yrs         1120 non-null   object
 7   payment_pcnt_income       1120 non-null   int64 
 8   gender_status             1120 non-null   object
 9   other_signators           1120 non-null   object
 10  time_in_residence         1120 non-null   int64 
 11  property                  1120 non-null   object
 12  age_yrs                   1120 non-null   int64 
 13  other_credit_outstanding  1120 non-null   object
 14  home_ownership          

## 4. Visualización de datos

### Variables de entrada

In [87]:
#Lista de variables categóricas
catCols = d.select_dtypes(include = ["object", 'category']).columns.tolist()

d[catCols].head(2)

,checking_account_status,credit_history,purpose,savings_account_balance,time_employed_yrs,gender_status,other_signators,property,other_credit_outstanding,home_ownership,job_category,telephone,foreign_worker
0,none,critical account - other non-bank loans,radio/television,< 100 DM,>= 7 years,male-single,none,car or other,none,own,skilled,none,yes
1,none,current loans paid,car,500 - 1000 DM,1 - 4 years,male-single,none,building society savings/life insurance,none,rent,skilled,none,yes


In [88]:
#Lista de variables numéricas

numCols=d.select_dtypes(include = ['float64','float64','int32','int64']).columns.tolist()

d[numCols].head(2)

,loan_duration_mo,loan_amount,payment_pcnt_income,time_in_residence,age_yrs,number_loans,dependents,bad_credit
0,28,2743,4,2,29,2,1,0
1,6,1236,2,4,50,1,1,0


In [89]:
##Visualización de frecuencia de instancias para variables categóricas
#multiple_plot(3, d , catCols, None, 'countplot', 'Frecuencia de instancias para variables categóricas',30)

In [90]:
##Visualización de variables numéricas
#multiple_plot(1, d , numCols, None, 'scatterplot', 'Relación entre las variables numéricas',30)

In [91]:
#Eliminar la variable de salida de la lista de variable numéricas
numCols.remove('bad_credit')

### Variable de salida

In [92]:
# Distriución de la variable de salida

d.groupby('bad_credit').bad_credit.count().sort_values(ascending=False)

,bad_credit
bad_credit,
0,560
1,560


In [93]:
##Visualización de la variable de salida
#multiple_plot(1, d , None, 'bad_credit', 'countplot', 'Gráfica de frecuencia de bad Credit',0)

## 5. Transformación de datos

### Creación de variables Dummies

In [94]:
# Aplicación de la función de usuario Dummies: one-hot encoding

d =pd.get_dummies(d, drop_first=1)

d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1120 entries, 0 to 1119
Data columns (total 44 columns):
 #   Column                                                  Non-Null Count  Dtype
---  ------                                                  --------------  -----
 0   loan_duration_mo                                        1120 non-null   int64
 1   loan_amount                                             1120 non-null   int64
 2   payment_pcnt_income                                     1120 non-null   int64
 3   time_in_residence                                       1120 non-null   int64
 4   age_yrs                                                 1120 non-null   int64
 5   number_loans                                            1120 non-null   int64
 6   dependents                                              1120 non-null   int64
 7   bad_credit                                              1120 non-null   int64
 8   checking_account_status_< 0 DM                          11

## 6. Creación del modelo

### Dividir el conjunto de datos

In [95]:
#Se establece las variables de entrada 'X' y la variable de salida 'y'

X = d.drop(columns ='bad_credit')
y = d['bad_credit']

# la validación cruzada se realiza sobre todo el dataset
X_Completo = X
y_Completo = y

### Escalar Variables

In [96]:
#Se establecen las variables numéricas a escalar

#Num_vars se le asigna la lista con las variables numerivas para posteriormente escalarlas
num_vars = numCols

print(num_vars)

['loan_duration_mo', 'loan_amount', 'payment_pcnt_income', 'time_in_residence', 'age_yrs', 'number_loans', 'dependents']


In [97]:
#Se establecen las variables numéricas a escalar

#Num_vars se le asigna la lista con las variables numerivas para posteriormente escalarlas
num_vars = numCols

# Se crea un objeto MinMaxScaler
scaler = MinMaxScaler()

# Se escalan los valores del dataset entrenamiento y prueba de las columnas numéricas
X_Completo[num_vars] = scaler.fit_transform(X_Completo[num_vars])

X_Completo[num_vars].head()

,loan_duration_mo,loan_amount,payment_pcnt_income,time_in_residence,age_yrs,number_loans,dependents
0,0.352941,0.137174,1.000000,0.333333,0.178571,0.333333,0.0
1,0.029412,0.054253,0.333333,1.000000,0.553571,0.000000,0.0
2,0.470588,0.112743,1.000000,1.000000,0.357143,0.000000,0.0
3,0.235294,0.374546,0.666667,1.000000,0.303571,0.333333,1.0
4,0.250000,0.109552,1.000000,0.333333,0.553571,0.333333,0.0


In [98]:
# Guardar el scaler

SCALER_FILE_NAME = 'minmaxFull_GermanCredits.pkl'
local_scaler_path = os.path.join(MODELS_SCALER_PATH, SCALER_FILE_NAME)

joblib.dump(scaler,  local_scaler_path)

['modelos/scaler/minmaxFull_GermanCredits.pkl']

### Creación del modelo

#### 6.1 Grid Search basado en out-of-bag score

In [60]:
# Definición del grid de parámetros
param_grid = ParameterGrid(
      {
      'n_estimators': [150, 200, 250],
      'max_features': ['sqrt', 'log2', 5, 7, 9],
      'max_depth': [5, 10, 15, None],
      'criterion': ['gini'] #, 'entropy'
      }
)

#Dicionario de resultados
resultados = {
    'params': [],
    'oob_accuracy': [],
    'train_accuracy': [],
    'gap_oob_train': []
}

#Ciclo de entrenamiento
for params in param_grid:

    modelo = RandomForestClassifier(
        bootstrap=True,
        oob_score=True,
        n_jobs=-1,
        random_state=123,
        **params
    )

    modelo.fit(X_Completo, y_Completo)

    # Accuracy entrenamiento
    train_acc = modelo.score(X_Completo, y_Completo)

    # OOB
    oob_acc = modelo.oob_score_

    # Gap (proxy de sobreajuste)
    gap = train_acc - oob_acc

    resultados['params'].append(params)
    resultados['oob_accuracy'].append(oob_acc)
    resultados['train_accuracy'].append(train_acc)
    resultados['gap_oob_train'].append(gap)

print("Grid Search con OOB completado.")


Grid Search con OOB completado.


In [61]:
# DataFrame
resultadosDf = pd.DataFrame(resultados)
resultadosDf = pd.concat(
    [resultadosDf, resultadosDf['params'].apply(pd.Series)], axis=1
)
resultadosDf = resultadosDf.drop(columns='params')


# Orden: mejor generalización + menor overfitting
resultadosDf = resultadosDf.sort_values(
    by=['oob_accuracy', 'gap_oob_train'],
    ascending=[False, True])

In [62]:
#Se eliminan aquellos que tienen más de 0.05 de gab entre train y test
resultados_filtrados = resultadosDf[resultadosDf['gap_oob_train'] < 0.07]
resultados_filtrados.head(10)

,oob_accuracy,train_accuracy,gap_oob_train,criterion,max_depth,max_features,n_estimators
9,0.779464,0.832143,0.052679,gini,5.0,7,150
5,0.777679,0.833036,0.055357,gini,5.0,log2,250
8,0.777679,0.833036,0.055357,gini,5.0,5,250
4,0.777679,0.835714,0.058036,gini,5.0,log2,200
7,0.777679,0.835714,0.058036,gini,5.0,5,200
11,0.775000,0.832143,0.057143,gini,5.0,7,250
3,0.775000,0.835714,0.060714,gini,5.0,log2,150
6,0.775000,0.835714,0.060714,gini,5.0,5,150
0,0.770536,0.827679,0.057143,gini,5.0,sqrt,150
10,0.767857,0.826786,0.058929,gini,5.0,7,200


In [63]:
resultados_filtrados.head(10)

,oob_accuracy,train_accuracy,gap_oob_train,criterion,max_depth,max_features,n_estimators
9,0.779464,0.832143,0.052679,gini,5.0,7,150
5,0.777679,0.833036,0.055357,gini,5.0,log2,250
8,0.777679,0.833036,0.055357,gini,5.0,5,250
4,0.777679,0.835714,0.058036,gini,5.0,log2,200
7,0.777679,0.835714,0.058036,gini,5.0,5,200
11,0.775000,0.832143,0.057143,gini,5.0,7,250
3,0.775000,0.835714,0.060714,gini,5.0,log2,150
6,0.775000,0.835714,0.060714,gini,5.0,5,150
0,0.770536,0.827679,0.057143,gini,5.0,sqrt,150
10,0.767857,0.826786,0.058929,gini,5.0,7,200


In [64]:
# Obtener los mejores parámetros del DataFrame de resultados OOB
best_oob_params_row = resultados_filtrados.iloc[0]
best_oob_hyperparameters = {
    'criterion': best_oob_params_row['criterion'],
    'max_depth': int(best_oob_params_row['max_depth']) if pd.notna(best_oob_params_row['max_depth']) else None,
    'max_features': best_oob_params_row['max_features'],
    'n_estimators': best_oob_params_row['n_estimators']
}


modelo.set_params(**best_oob_hyperparameters)
modelo.fit(X_Completo, y_Completo)

print("model entrenado con los mejores parámetros de los resultados OOB.")
modelo

model entrenado con los mejores parámetros de los resultados OOB.


RandomForestClassifier(max_depth=5, max_features=7, n_estimators=np.int64(150),
                       n_jobs=-1, oob_score=True, random_state=123)

In [65]:
modelo.score(X_Completo, y_Completo)

0.8321428571428572

In [66]:
#Se guarda el modelo de Random Forest

MODEL_FILE_NAME = 'RForest_oob.pkl'
local_model_path = os.path.join(MODELS_CLASIFICACION_PATH, MODEL_FILE_NAME)

joblib.dump(modelo,  local_model_path)

['modelos/clasificacion/RForest_oob.pkl']

### 8.2 Grid Search basado en validación cruzada

In [67]:
#Definicion del modelo
modelrf = RandomForestClassifier(bootstrap=True, random_state = 123)


# Grid de hiperparámetros evaluados

grid_param = {'n_estimators': [150, 200, 250],
                          'max_features': ['sqrt', 'log2', 5, 7, 9],
                          'max_depth'   : [5, 10, 15, None],
                          'criterion'   : ['gini'] #entropy
                         }


# definicion de la variable con el número de pliegues
CV = 10

# las metricas sobre las se evaluará el modelo
scoring = 'f1'

# Búsqueda por grid search con validación cruzada
grid_rf = RandomizedSearchCV (
                        estimator  = modelrf,
                        param_distributions = grid_param,
                        scoring    = scoring,
                        n_iter=30,  # en lugar de 80 combinaciones
                        cv = CV,
                        n_jobs     = - 1, #Usa todos los núcleos disponibles del CPU
                        #refit      = True, #entrena un modelo final automáticamente y puedes usar directamente grid_rf.predict()
                        return_train_score = True
                       )

grid_rf.fit(X_Completo, y_Completo)

RandomizedSearchCV(cv=10, estimator=RandomForestClassifier(random_state=123),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'criterion': ['gini'],
                                        'max_depth': [5, 10, 15, None],
                                        'max_features': ['sqrt', 'log2', 5, 7,
                                                         9],
                                        'n_estimators': [150, 200, 250]},
                   return_train_score=True, scoring='f1')

In [68]:
# Obtener los resultados del grid search
results_grid_rf = pd.DataFrame(grid_rf.cv_results_)

# Seleccionar las columnas deseadas
columns_grid_rf = ['param_criterion'] + \
                ['param_max_depth'] + \
                ['param_max_features'] + \
                ['param_n_estimators'] + \
                ['mean_test_score', 'std_test_score', 'mean_train_score', 'std_train_score']  + \
                [f'split{i}_test_score' for i in range(CV)]

# Filtrar y mostrar los resultados
results_grid_rf_filtered = results_grid_rf[columns_grid_rf]


# Crear la columna diff_test_train: mean_test_score - mean_train_score
results_grid_rf_filtered['diff_test_train'] = abs(
    results_grid_rf_filtered['mean_test_score'] -
    results_grid_rf_filtered['mean_train_score']
)


In [69]:

# Evitar sobreajuste: Filtrar para mostrar solo los resultados donde diff_test_train es menor o igual al 5%
results_grid_rf_filtered = results_grid_rf_filtered[results_grid_rf_filtered['diff_test_train'] <= 0.07]

# Verificar si results_grid_rf_filtered está vacío antes de continuar
if results_grid_rf_filtered.empty:
    print("No se identificaron valores")
else:
    # Encuentra el índice del máximo valor en la columna mean_test_score
    indice_min_diff_test_train = results_grid_rf_filtered['mean_test_score'].idxmax()

# Mostrar los scores promedios por cada parámetro
results_grid_rf_filtered[['param_criterion','param_max_depth','param_max_features','param_n_estimators', 'mean_test_score', 'std_test_score','mean_train_score','std_train_score', 'diff_test_train']]

,param_criterion,param_max_depth,param_max_features,param_n_estimators,mean_test_score,std_test_score,mean_train_score,std_train_score,diff_test_train
1,gini,5,log2,200,0.780651,0.032590,0.836957,0.003370,0.056305
15,gini,5,9,150,0.771215,0.039363,0.838964,0.006777,0.067749
18,gini,5,log2,150,0.777956,0.032998,0.835341,0.004896,0.057386
19,gini,5,sqrt,200,0.774102,0.040678,0.831220,0.005553,0.057117


In [70]:
# Top 3 de los mejores scores
results_grid_rf_filtered.sort_values(by='mean_test_score', ascending=False).head(3)[
    ['param_criterion','param_max_depth','param_max_features','param_n_estimators'
     , 'mean_train_score', 'mean_test_score', 'std_test_score', 'diff_test_train']]

,param_criterion,param_max_depth,param_max_features,param_n_estimators,mean_train_score,mean_test_score,std_test_score,diff_test_train
1,gini,5,log2,200,0.836957,0.780651,0.032590,0.056305
18,gini,5,log2,150,0.835341,0.777956,0.032998,0.057386
19,gini,5,sqrt,200,0.831220,0.774102,0.040678,0.057117


In [71]:
# Seleccionar el registro correspondiente al índice máximo
registro_min_diff_test_train = results_grid_rf_filtered.loc[indice_min_diff_test_train]

registro_min_diff_test_train.transpose()

,1
param_criterion,gini
param_max_depth,5
param_max_features,log2
param_n_estimators,200
mean_test_score,0.780651
std_test_score,0.03259
mean_train_score,0.836957
std_train_score,0.00337
split0_test_score,0.774775
split1_test_score,0.781818


In [72]:
results_grid_rf_filtered.loc[indice_min_diff_test_train]['mean_test_score']

np.float64(0.7806514268213249)

In [73]:
# Usar los mejores parámetros para crear el modelo
modelrf.set_params(criterion = results_grid_rf_filtered.loc[indice_min_diff_test_train]['param_criterion']
                        , max_depth = results_grid_rf_filtered.loc[indice_min_diff_test_train]['param_max_depth']
                        , max_features = results_grid_rf_filtered.loc[indice_min_diff_test_train]['param_max_features']
                        , n_estimators = results_grid_rf_filtered.loc[indice_min_diff_test_train]['param_n_estimators'])
modelrf.fit(X_Completo, y_Completo)

RandomForestClassifier(max_depth=5, max_features='log2',
                       n_estimators=np.int64(200), random_state=123)

### Guardar modelo

In [74]:
#Se guarda el modelo de Random Forest

MODEL_FILE_NAME = 'RForest_CV.pkl'
local_model_path = os.path.join(MODELS_CLASIFICACION_PATH, MODEL_FILE_NAME)

joblib.dump(modelrf,  local_model_path)


['modelos/clasificacion/RForest_CV.pkl']

### Importancia de predictores

#### Importancia por pureza de nodos

In [75]:
importancia_predictores = pd.DataFrame(
                            {'predictor': X_Completo.columns,
                             'importancia': modelrf.feature_importances_}
                            )
print("Importancia de los predictores en el modelo")
print("-------------------------------------------")
importancia_predictores.sort_values('importancia', ascending=False).head(10)

Importancia de los predictores en el modelo
-------------------------------------------


,predictor,importancia
9,checking_account_status_none,0.155536
1,loan_amount,0.104684
0,loan_duration_mo,0.103084
4,age_yrs,0.083816
7,checking_account_status_< 0 DM,0.063268
10,credit_history_critical account - other non-ba...,0.053754
32,property_real estate,0.031664
21,savings_account_balance_unknown/none,0.029151
23,time_employed_yrs_< 1 year,0.026980
19,savings_account_balance_< 100 DM,0.019433


### Predicción de probabilidades

In [76]:
# Predicción de probabilidades
# ==============================================================================
predicciones = modelrf.predict_proba(X = X_Completo)
predicciones[:5, :]

array([[0.76411457, 0.23588543],
       [0.69348823, 0.30651177],
       [0.69231569, 0.30768431],
       [0.63787052, 0.36212948],
       [0.69830069, 0.30169931]])

In [77]:
# Clasificación empleando la clase de mayor probabilidad
# ==============================================================================
df_predicciones = pd.DataFrame(data=predicciones, columns=['0', '1'])
df_predicciones['clasificacion_default_0.5'] = np.where(df_predicciones['0'] > df_predicciones['1'], 0, 1)
df_predicciones.head(5)

,0,1,clasificacion_default_0.5
0,0.764115,0.235885,0
1,0.693488,0.306512,0
2,0.692316,0.307684,0
3,0.637871,0.362129,0
4,0.698301,0.301699,0


In [78]:
# Clasificación final empleando un threshold de 0.8 para la clase 1.
# ==============================================================================
df_predicciones['clasificacion_custom_0.8'] = np.where(df_predicciones['1'] > 0.8, 1, 0)
df_predicciones.iloc[4:10, :]

,0,1,clasificacion_default_0.5,clasificacion_custom_0.8
4,0.698301,0.301699,0,0
5,0.464088,0.535912,1,0
6,0.642613,0.357387,0,0
7,0.668115,0.331885,0,0
8,0.642992,0.357008,0,0
9,0.662366,0.337634,0,0


#### Referencias


- Random Forest
    - https://www.cienciadedatos.net/documentos/py08_random_forest_python.html
    - https://fhernanb.github.io/libro_mod_pred/rand-forests.html
    - https://quantdare.com/decision-trees-gini-vs-entropy/
    